# Five-Fold Cross-Validation for Vestibular Schwannoma Segmentation

Trains and evaluates three 3D segmentation models (UNet, DynUNet, SegMamba) on vestibular schwannoma (VS) MRI with 5-fold cross-validation, using the fastMONAI patch workflow. VS is a benign tumor of the vestibulocochlear nerve; accurate delineation on Contrast-Enhanced T1-weighted (CE-T1w) MRI supports treatment planning and volumetric follow-up ([Kujawa et al., 2024](https://doi.org/10.3389/fncom.2024.1365727); [Dhayalan et al., 2023](https://doi.org/10.1001/jama.2023.12222)).

## Task and data

Binary segmentation: label every voxel tumor (1) or background (0). We use 346 CE-T1w cases pooled from the Queen Square cohort ([Shapey et al., 2021](https://doi.org/10.1038/s41597-021-01064-w)) and the crossMoDA challenge ([Dorent et al., 2023](https://doi.org/10.1016/j.media.2022.102628)), each with a ground-truth mask. Tumors are small relative to the field of view, so we train on patches sampled around the lesion.

## Cross-validation and models

The 346 cases are split into five folds via the `fold` column (1..5) of the dataset CSV; each run holds out one fold for validation and trains on the other four, so every case is validated once. A fixed column keeps the folds identical across models and reproducible; the `split` column is ignored.

- **UNet** (MONAI): convolutional encoder-decoder baseline.
- **DynUNet** (MONAI): nnU-Net-style UNet ([Isensee et al., 2024](https://doi.org/10.1007/978-3-031-72114-4_47)) with residual blocks and deep supervision.
- **SegMamba** (our fork): a state-space backbone with linear-time selective scan for long-range 3D context.

All three run through an identical data, loss, and evaluation pipeline, so score differences reflect the model, not the plumbing. Comparing a CNN baseline, an nnU-Net-style model, and a state-space model probes whether newer architectures beat a standard CNN on VS ([Connor et al., 2025](https://doi.org/10.5152/iao.2025.241693); [Häußler et al., 2025](https://doi.org/10.1002/lary.31979)). References are listed at the end.

## 1. Environment setup

In [ ]:
import os
import gc
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchio as tio

from monai.losses import DiceCELoss, DeepSupervisionLoss
from monai.networks.layers import Norm
from monai.networks.nets import UNet, DynUNet

from fastMONAI.vision_all import *

# Optional fork; import here so a missing fork is reported now with the install hint, not mid-sweep.
_SEGMAMBA_INSTALL_HINT = (
    "  GPU (training):  pip install 'segmamba-v2[gpu] @ git+https://github.com/skaliy/SegMamba-V2.git'\n"
    "  CPU (inference): pip install 'segmamba-v2[cpu] @ git+https://github.com/skaliy/SegMamba-V2.git'"
)
try:
    from models_segmamba.segmambav2 import SegMamba
    SEGMAMBA_AVAILABLE = True
except ImportError as exc:
    SegMamba, SEGMAMBA_AVAILABLE = None, False
    print(f"[segmamba] fork not available ({exc}); UNet and DynUNet still run. To enable it:\n"
          + _SEGMAMBA_INSTALL_HINT)

# Resolve paths against the repo root so they work from any launch dir.
import fastMONAI
REPO_ROOT = Path(fastMONAI.__file__).resolve().parent.parent
os.chdir(REPO_ROOT / "research" / "vestibular_schwannoma")

## 2. Run configuration

A full run is `3 models x 5 folds x 500 epochs`, on the order of days on one modern GPU. Shrink the knobs (e.g. `MODELS_TO_RUN = ["unet"]`, `FOLDS_TO_RUN = [1]`, `EPOCHS = 2`) for a quick smoke test that still exercises every stage. `TARGET_SPACING` and `PATCH_SIZE` are the shared preprocessing contract and must match `02_inference_new_cases.ipynb`.

In [ ]:
MODELS_TO_RUN = ["unet", "dynunet", "segmamba"]
FOLDS_TO_RUN  = [1, 2, 3, 4, 5]

EPOCHS  = 500        # paper setting; 2 for a quick demo
BS      = 4
LR      = 1e-3       # max LR for fit_one_cycle
USE_TTA = True       # 8-flip TTA at evaluation

# Shared preprocessing contract. MUST match 02_inference_new_cases.ipynb.
TARGET_SPACING = [0.4102, 0.4102, 1.5]   # resample target, voxel size in mm
PATCH_SIZE     = [192, 192, 48]          # patch shape in voxels

DATA_CSV = "ml_dataset.csv"

# cuDNN autotuner: fixed patch size means the fastest kernels are found once and reused.
torch.backends.cudnn.benchmark = True

print(f"Models: {MODELS_TO_RUN}")
print(f"Folds : {FOLDS_TO_RUN}")
print(f"Epochs: {EPOCHS} | Batch size: {BS} | LR: {LR} | TTA: {USE_TTA}")
print(f"Target spacing: {TARGET_SPACING} | Patch size: {PATCH_SIZE}")

## 3. Dataset and folds

The CSV lists one row per case with raw image and mask paths, the pre-assigned `fold` (1..5), and a `split` column. `train_one_fold` sets the validation set as `is_val = (fold == fold_num)`; all 346 cases take part, so the `split` column is not used here.

In [ ]:
train_df = pd.read_csv(DATA_CSV)
print(f"Total cases: {len(train_df)}")

print("\nCases per fold (each fold is the validation set when held out):")
print(train_df["fold"].value_counts().sort_index().to_string())

print("\n'split' column (present in the CSV but IGNORED by cross-validation):")
print(train_df["split"].value_counts().to_string())

train_df[["case_id", "t1_img_path", "t1_seg_path", "fold", "split"]].head()

## 4. Preprocess once to disk

Preprocessing (RAS+ reorientation, resampling to `TARGET_SPACING`, foreground-masked Z-normalization) is fold-independent, so we run it once over all 346 cases. `preprocess_dataset` writes processed volumes under `preprocessed/` and adds `t1_img_path_preprocessed` / `t1_seg_path_preprocessed` to `train_df` in place; `skip_existing=True` makes it idempotent, reusing the cache on re-run. During training, `PatchConfig(preprocessed=True)` then skips reorder, resample, and normalization.

### Intensity normalization

`ZNormalization(masking_method="foreground")` computes mean and std from foreground voxels (intensity above zero, read from the image not the mask) and applies them to the whole volume. Restricting the statistics to the foreground stops the large zero-valued background from washing out tissue contrast.

In [ ]:
# Single source of truth for pre-patch / pre-inference intensity normalization.
pre_patch_tfms = [ZNormalization(masking_method="foreground")]

preprocess_dataset(
    train_df,
    img_col="t1_img_path",
    mask_col="t1_seg_path",
    output_dir="preprocessed",
    target_spacing=TARGET_SPACING,
    transforms=pre_patch_tfms,
    max_workers=32,
)

# The _preprocessed columns are written for every row, including cases that failed, so an
# unchecked failure resurfaces much later as a missing file deep inside training.
missing = [(cid, p)
           for col in ("t1_img_path_preprocessed", "t1_seg_path_preprocessed")
           for cid, p in zip(train_df.case_id, train_df[col]) if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} of {2 * len(train_df)} preprocessed files are missing, so those "
        f"cases failed preprocessing. First few: {missing[:5]}")

print("Added columns:", [c for c in train_df.columns if c.endswith("_preprocessed")])
train_df[["t1_img_path", "t1_img_path_preprocessed"]].head()

## 5. Shared pipeline building blocks

Every model trains through the same helpers, so result differences come from the network, not the pipeline. `create_patch_config()` returns the `PatchConfig` that governs how patches are sampled and stitched:

- A **label sampler** with `label_probabilities={0: 0.2, 1: 0.8}` draws 80% of patches centered on tumor voxels; without this bias the network would rarely see foreground.
- `preprocessed=True` tells the loader the volumes on disk are already reoriented, resampled, and normalized, so it does not repeat that work.
- `normalization=pre_patch_tfms` records the intensity normalization on the config as the single source of truth: not re-applied under `preprocessed=True`, but logged to MLflow and used to rebuild the identical transform at inference.
- `keep_largest_component=True` post-processes each prediction to its single largest connected region, which suits a solitary VS tumor.

In [ ]:
def create_patch_config():
    """Patch sampling and aggregation settings shared by all three models."""
    return PatchConfig(
        patch_size=PATCH_SIZE,
        samples_per_volume=4,
        sampler_type="label",
        label_probabilities={0: 0.2, 1: 0.8},
        patch_overlap=0.5,
        keep_largest_component=True,
        target_spacing=TARGET_SPACING,
        preprocessed=True,
        normalization=pre_patch_tfms,
        aggregation_mode="hann",
        # queue_length patches stay resident in RAM (~17 GB at this patch size). Sized for
        # a large host; lower both knobs on a smaller machine.
        queue_num_workers=16,
        queue_length=1200,
    )


patch_config = create_patch_config()

# Persist the config so 02_inference_new_cases.ipynb rebuilds identical preprocessing and patch
# settings. Local copy here; train_one_fold also logs it as a per-run MLflow artifact.
store_patch_variables(
    "inference_patch_config.json",
    patch_size=patch_config.patch_size,
    patch_overlap=patch_config.patch_overlap,
    aggregation_mode=patch_config.aggregation_mode,
    apply_reorder=patch_config.apply_reorder,
    target_spacing=patch_config.target_spacing,
    sampler_type=patch_config.sampler_type,
    label_probabilities=patch_config.label_probabilities,
    samples_per_volume=patch_config.samples_per_volume,
    queue_length=patch_config.queue_length,
    queue_num_workers=patch_config.queue_num_workers,
    keep_largest_component=patch_config.keep_largest_component,
    normalization=patch_config.normalization,
)
patch_config

### GPU augmentation

`create_gpu_augmentation()` returns a `GpuPatchAugmentation` that augments each batch on the GPU so the input pipeline is not the bottleneck. Transforms and probabilities follow nnU-Net conventions; affine translations are given in voxels, so millimeter shifts are divided by the target spacing, and augmentation is applied to the training set only.

In [ ]:
def create_gpu_augmentation():
    """nnU-Net-inspired GPU-batched augmentation applied to training patches."""
    ts = TARGET_SPACING
    return GpuPatchAugmentation(
        affine={
            "scales": (0.7, 1.4),
            "degrees": (5, 5, 30),
            "translation": (25 / ts[0], 25 / ts[1], 5 / ts[2]),
            "default_pad_value": 0.0,
            "p": 0.2,
        },
        anisotropy={"axes": (0, 1, 2), "downsampling": (2, 4), "p": 0.25},
        flip={"axes": (0, 1, 2), "p": 0.5},
        gamma={"log_gamma": (-0.3, 0.3), "p": 0.3},
        intensity_scale={"scale_range": (0.75, 1.25), "p": 0.1},
        noise={"std": 0.1, "p": 0.1},
        blur={"std": (0.5, 1.0), "p": 0.2},
    )

### Sanity check: inspect a training batch

Draw one batch of patches from the first fold and overlay the ground-truth mask, a quick check that the label sampler centers patches on the tumor, the intensities look normalized, and the mask lines up with the lesion. The cell is standalone and does not touch `train_one_fold` or the driver.

In [ ]:
sanity_fold = FOLDS_TO_RUN[0]
sanity_df = train_df.copy()
sanity_df["is_val"] = sanity_df["fold"] == sanity_fold

sanity_config = create_patch_config()            # fresh config leaves patch_config alone
sanity_config.samples_per_volume = 1
sanity_config.queue_length = BS
sanity_config.queue_num_workers = 2

# No gpu_augmentation: show clean patches. preprocessed=True means they carry the on-disk
# foreground Z-norm, exactly what the model trains on.
sanity_dls = MedPatchDataLoaders.from_df(
    df=sanity_df,
    img_col="t1_img_path_preprocessed",
    mask_col="t1_seg_path_preprocessed",
    valid_col="is_val",
    patch_config=sanity_config,
    bs=BS,
)

sanity_dls.show_batch(dl_idx=0, max_n=4, overlay=True, anatomical_plane=2)

sanity_dls.close()

## 6. Model definitions

The three models are registered in a single `MODELS` dictionary. Each entry provides a `make_model` factory, a `make_loss` factory, and the experiment name, checkpoint filename, and results directory the inference notebook expects:

| key | experiment | checkpoint | results dir |
|-----|------------|-----------|-------------|
| `unet` | `vs5f_unet` | `best_unet` | `cv_results_unet` |
| `dynunet` | `vs5f_dynunet` | `best_dynunet` | `cv_results_dynunet` |
| `segmamba` | `vs5f_segmamba` | `best_segmamba` | `cv_results_segmamba` |

UNet and SegMamba use a Dice + cross-entropy loss. DynUNet emits predictions at several decoder depths, so it is wrapped in `DynUNetDSAdapter` and trained with `DeepSupervisionLoss`. UNet and DynUNet are compiled with `torch.compile`; SegMamba is not, because it relies on custom CUDA kernels.

In [ ]:
MODELS = {}

def make_unet():
    model = UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        channels=(64, 128, 256, 512, 1024),
        strides=(2, 2, 2, 2),
        num_res_units=4,
        norm=Norm.INSTANCE,
        act=("LEAKYRELU", {"negative_slope": 0.01, "inplace": True}),
    )
    return torch.compile(model)


def make_unet_loss():
    return CustomLoss(loss_func=DiceCELoss(
        to_onehot_y=True, softmax=True, include_background=False, batch=True
    ))


MODELS["unet"] = dict(
    make_model=make_unet, make_loss=make_unet_loss,
    experiment="vs5f_unet", best_fname="best_unet", results_dir="cv_results_unet",
)

def make_dynunet():
    dynunet = DynUNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        kernel_size=[[3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3]],
        strides=[[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]],
        upsample_kernel_size=[[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]],
        filters=[64, 128, 256, 512, 1024],
        res_block=True,
        deep_supervision=True,
        deep_supr_num=3,
    )
    model = DynUNetDSAdapter(dynunet)
    return torch.compile(model)


def make_dynunet_loss():
    base_loss = DiceCELoss(
        to_onehot_y=True, softmax=True, include_background=False, batch=True
    )
    return CustomLoss(loss_func=DeepSupervisionLoss(base_loss, weight_mode="exp"))


MODELS["dynunet"] = dict(
    make_model=make_dynunet, make_loss=make_dynunet_loss,
    experiment="vs5f_dynunet", best_fname="best_dynunet", results_dir="cv_results_dynunet",
)

# SegMamba (optional fork): registered only if it imported cleanly. No torch.compile (custom CUDA kernels).
if SEGMAMBA_AVAILABLE:

    def make_segmamba():
        return SegMamba(
            in_chans=1,
            out_chans=2,
            depths=[2, 2, 2, 2],
            feat_size=[48, 96, 192, 384],
            hidden_size=768,
        )

    def make_segmamba_loss():
        return CustomLoss(loss_func=DiceCELoss(
            to_onehot_y=True, softmax=True, include_background=False, batch=True
        ))

    MODELS["segmamba"] = dict(
        make_model=make_segmamba, make_loss=make_segmamba_loss,
        experiment="vs5f_segmamba", best_fname="best_segmamba",
        results_dir="cv_results_segmamba",
    )

print("Registered models:", list(MODELS))

# Note a requested-but-unavailable segmamba (setup cell already printed the install hint).
if "segmamba" in MODELS_TO_RUN and "segmamba" not in MODELS:
    print("\n[segmamba] requested but its fork is not importable, so it is skipped "
          "(see the install hint from the setup cell above).")

## 7. Train one fold

`train_one_fold(model_key, fold_num, train_df, patch_config)` runs one training job: it marks the held-out fold as validation, builds the patch DataLoaders from the preprocessed columns, constructs the model and loss from the registry, and trains with `fit_one_cycle`. Training uses:

- `AccumulatedDice(n_classes=2)` as the monitored metric (nnU-Net-style pseudo-Dice accumulated over the whole validation set).
- `EMACheckpoint` to save the best weights by the EMA of that metric, under the per-model checkpoint name.
- `create_mlflow_callback` to log parameters, metrics, the split, and artifacts to the per-model MLflow experiment.
- `.to_bf16()` for bfloat16 mixed-precision training.

At the end it calls `evaluate_fold` (next section) and frees GPU memory.

In [ ]:
def train_one_fold(model_key, fold_num, train_df, patch_config):
    """Train one model on one fold, then evaluate the held-out fold."""
    entry = MODELS[model_key]
    print(f"\n{'='*60}")
    print(f"  {model_key.upper()}  |  FOLD {fold_num}")
    print(f"{'='*60}\n")

    # Held-out fold becomes the validation set.
    train_df = train_df.copy()
    train_df["is_val"] = train_df["fold"] == fold_num

    # Content fingerprint of the label set, logged to MLflow for reproducibility.
    med_dataset = MedDataset(
        img_list=train_df.t1_seg_path.tolist(), dtype=MedMask, max_workers=32
    )

    gpu_aug = create_gpu_augmentation()

    # Normalization lives on patch_config; preprocessed columns are already normalized, so
    # preprocessed=True skips re-applying it here.
    dls = MedPatchDataLoaders.from_df(
        df=train_df,
        img_col="t1_img_path_preprocessed",
        mask_col="t1_seg_path_preprocessed",
        valid_col="is_val",
        patch_config=patch_config,
        gpu_augmentation=gpu_aug,
        bs=BS,
    )

    print(f"[{model_key} fold {fold_num}] Train: {len(dls.train.subjects_dataset)}, "
          f"Val: {len(dls.valid.subjects_dataset)}")

    model = entry["make_model"]()
    loss_func = entry["make_loss"]()

    learn = Learner(dls, model, loss_func=loss_func,
                    metrics=[AccumulatedDice(n_classes=2)]).to_bf16()

    save_best = EMACheckpoint(
        monitor="accumulated_dice", momentum=0.9,
        comp=np.greater, fname=entry["best_fname"], with_opt=False,
    )

    mlflow_cb = create_mlflow_callback(
        learn,
        experiment_name=entry["experiment"],
        run_name=f"fold_{fold_num}",
        extra_tags={"fold": str(fold_num)},
        dataset_version=med_dataset.fingerprint,
    )

    learn.fit_one_cycle(EPOCHS, LR, cbs=[mlflow_cb, save_best])

    # Reopen this fold's run to attach the patch config, so 02 can pull it bound to these
    # checkpoints. The callback closed the run but kept its id.
    import mlflow
    if mlflow_cb.run_id is not None:
        with mlflow.start_run(run_id=mlflow_cb.run_id):
            mlflow.log_artifact("inference_patch_config.json", artifact_path="config")

    results_df = evaluate_fold(
        learn, patch_config, dls, pre_patch_tfms, fold_num,
        tta=USE_TTA, mlflow_cb=mlflow_cb, results_dir=Path(entry["results_dir"]),
    )

    # Release the queue workers and GPU memory before the next (model, fold).
    dls.close()
    del learn, model, dls, gpu_aug
    torch.cuda.empty_cache()
    gc.collect()

    return results_df

## 8. Evaluate a fold

After training, we score the held-out fold on the raw validation images (not the preprocessed copies), re-applying the identical `ZNormalization` through `pre_inference_tfms`, which mirrors real inference end to end. For each case we compute Dice, sensitivity, precision, lesion-detection rate, and signed relative volume error, plus spacing-aware surface metrics (ASSD, HD95, and Normalized Surface Dice at 0.5/1.0/2.0 mm) via `calculate_surface_metrics`. Voxel spacing is read per case from the ground-truth file, because the cohort spacing is not uniform.

In [ ]:
def evaluate_fold(learn, patch_config, dls, pre_patch_tfms, fold_num, tta, mlflow_cb, results_dir):
    results_dir = Path(results_dir)
    fold_dir = results_dir / f"fold_{fold_num}"
    pred_dir = fold_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)

    learn.cuda()

    val_df = dls._valid_source_df
    val_img_paths = val_df["t1_img_path"].tolist()
    val_mask_paths = val_df["t1_seg_path"].tolist()

    print(f"[Fold {fold_num}] Running inference on {len(val_img_paths)} validation images...")
    predictions = patch_inference(
        learner=learn,
        config=patch_config,
        file_paths=val_img_paths,
        pre_inference_tfms=pre_patch_tfms,
        save_dir=str(pred_dir),
        progress=True,
        tta=tta,
    )

    # Per-case metric table: reads each GT mask's data and spacing from one object so they can't disagree.
    results_df = evaluate_segmentations(predictions, val_mask_paths,
                                        case_ids=val_df["case_id"].tolist())
    results_df.insert(1, "image", [Path(p).name for p in val_img_paths])
    if results_df["spacing_mm"].astype(str).nunique() == 1:
        print(f"[Fold {fold_num}] WARNING: spacing_mm is constant across the fold "
              f"({results_df['spacing_mm'].iloc[0]}); verify per-case derivation if unexpected.")
    results_df.to_csv(fold_dir / "results.csv", index=False)

    # Benchmark provenance (arxiv:2410.02630 transparency): metric impl, NSD tolerances, per-case spacing.
    from fastMONAI.vision_metrics import _SURFACE_DISTANCE_SOURCE
    with open(fold_dir / "benchmark_meta.json", "w") as f:
        json.dump({
            "surface_distance_source": _SURFACE_DISTANCE_SOURCE,
            "nsd_tolerances_mm": [0.5, 1.0, 2.0],   # evaluate_segmentations defaults
            "nsd_headline_tau_mm": 1.0,
            "status_counts": results_df["surface_status"].value_counts().to_dict(),
            "per_case": [{"case_id": r["case_id"], "spacing_mm": list(r["spacing_mm"]),
                          "surface_status": r["surface_status"]}
                         for r in results_df.to_dict("records")],
        }, f, indent=2)

    # inf-safe means: one_empty cases score inf; replace with NaN so pandas skips it (MLflow rejects non-finite).
    numeric = results_df.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan)
    mlflow_cb.log_metrics_table(results_df, display=False)
    mlflow_cb.log_metrics({f"val_{m}": numeric[m].mean() for m in numeric.columns})
    mlflow_cb.log_dataframe(results_df)

    print(f"[Fold {fold_num}] Results saved to {fold_dir / 'results.csv'}")
    print(f"[Fold {fold_num}] DSC: {results_df['dsc'].mean():.4f} +/- {results_df['dsc'].std():.4f}")
    return results_df

## 9. Cross-validation driver

The driver sweeps `MODELS_TO_RUN x FOLDS_TO_RUN`, calling `train_one_fold` for each pair. Each pair is wrapped in try/except, so one failure (for example an out-of-memory error on one fold) is logged and the sweep continues rather than aborting.

In [ ]:
for model_key in MODELS_TO_RUN:
    if model_key not in MODELS:
        print(f"[skip] '{model_key}' is not registered (see the model definitions cell above).")
        continue

    Path(MODELS[model_key]["results_dir"]).mkdir(parents=True, exist_ok=True)

    for fold_num in FOLDS_TO_RUN:
        try:
            train_one_fold(model_key, fold_num, train_df, patch_config)
        except Exception as exc:
            print(f"[FAILED] {model_key} fold {fold_num}: {type(exc).__name__}: {exc}")
            traceback.print_exc()
            torch.cuda.empty_cache()
            gc.collect()
            continue

print("\nSweep complete.")

## 10. Aggregate per-model results

For each model, `aggregate_results` concatenates the per-fold `results.csv` files into `cv_summary.csv` and prints a per-metric mean/std summary plus the per-fold DSC breakdown. Millimeter surface distances (ASSD, HD95) are averaged over finite values only, since a case where exactly one of prediction or ground truth is empty scores `+inf` by design.

In [ ]:
def aggregate_results(results_dir):
    """Concatenate a model's per-fold results.csv into cv_summary.csv and print a summary; None if no results yet."""
    results_dir = Path(results_dir)
    all_results = []
    for fold_dir in sorted(results_dir.glob("fold_*")):
        csv_path = fold_dir / "results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["fold"] = int(fold_dir.name.split("_")[1])
            all_results.append(df)

    if not all_results:
        print(f"No fold results found in {results_dir}. Run training first.")
        return None

    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(results_dir / "cv_summary.csv", index=False)

    metrics = ["dsc", "sensitivity", "precision", "ldr", "rve",
               "assd_mm", "hd95_mm", "nsd_tau0.5_mm", "nsd_tau1.0_mm", "nsd_tau2.0_mm"]
    inf_excluded = {"assd_mm", "hd95_mm"}  # one_empty -> inf must not poison the mean
    print(f"\n{'='*60}")
    print(f"  CROSS-VALIDATION SUMMARY: {results_dir.name}")
    print(f"{'='*60}")
    print(f"  Folds completed: {sorted(combined['fold'].unique())}")
    print(f"  Total subjects:  {len(combined)}\n")
    print(f"  {'Metric':<15} {'Mean':>10} {'Std':>10}")
    print(f"  {'-'*35}")
    for m in metrics:
        if m not in combined.columns:
            continue
        col = combined[m]
        note = ""
        if m in inf_excluded:
            mask = np.isfinite(col)
            n_skip = int((~mask).sum())
            col = col[mask]
            if n_skip:
                note = f"  ({n_skip} non-finite excl.)"
        print(f"  {m:<15} {col.mean():>10.4f} {col.std():>10.4f}{note}")

    if "surface_status" in combined.columns:
        print(f"\n  Surface-metric status counts:")
        for status, n in combined["surface_status"].value_counts().items():
            print(f"    {status:<12} {n}")

    print(f"\n  Per-fold DSC:")
    for fold_num, group in combined.groupby("fold"):
        print(f"    Fold {fold_num}: {group['dsc'].mean():.4f} +/- {group['dsc'].std():.4f}")

    print(f"\n  Results saved to {results_dir / 'cv_summary.csv'}")
    return combined

In [ ]:
cv_combined = {}
for model_key in MODELS_TO_RUN:
    entry = MODELS.get(model_key)
    if entry is None:
        print(f"[skip] '{model_key}' is not registered.")
        continue
    combined = aggregate_results(Path(entry["results_dir"]))
    if combined is not None:
        cv_combined[model_key] = combined

## 11. Cross-model comparison

For every model that produced results, we pool all held-out cases across folds, report the mean and standard deviation of each key metric, save the numbers to `cv_model_comparison.csv`, and display a compact `mean +/- std` view. The same inf-safe averaging is applied, so the millimeter surface metrics ignore non-finite per-case scores (and NaN, e.g. RVE on an empty ground truth, is skipped the same way).

In [ ]:
KEY_METRICS = ["dsc", "sensitivity", "precision", "ldr", "rve",
               "assd_mm", "hd95_mm", "nsd_tau1.0_mm"]

rows = []
for model_key, combined in cv_combined.items():
    row = {"model": model_key,
           "folds": sorted(combined["fold"].unique().tolist()),
           "n_cases": int(len(combined))}
    for m in KEY_METRICS:
        if m in combined.columns:
            finite = combined[m].replace([np.inf, -np.inf], np.nan)  # drop inf; pandas skips NaN
            row[f"{m}_mean"] = float(finite.mean())
            row[f"{m}_std"] = float(finite.std())
    rows.append(row)

if rows:
    comparison_df = pd.DataFrame(rows).set_index("model")
    comparison_df.to_csv("cv_model_comparison.csv")
    print("Saved cross-model comparison to cv_model_comparison.csv\n")

    pretty = pd.DataFrame(index=comparison_df.index)
    for m in KEY_METRICS:
        mc, sc = f"{m}_mean", f"{m}_std"
        if mc in comparison_df.columns:
            pretty[m] = [f"{mu:.4f} +/- {sd:.4f}"
                         for mu, sd in zip(comparison_df[mc], comparison_df[sc])]
    display(pretty)
else:
    print("No per-model results available yet. Run the driver loop first.")

## 12. Qualitative check: prediction vs. ground truth

A visual sanity check on the saved validation masks. We reload one model's predictions from disk and compare the median-DSC (representative) and worst-DSC (failure) cases against their ground truth; input, ground truth, and prediction share each case's original voxel grid, so they line up slice for slice.

In [ ]:
from fastMONAI.vision_plot import *
import matplotlib.pyplot as plt


def _pred_filename(input_path):
    """Prediction filename patch_inference wrote: '<stem>_pred.nii[.gz]'. Mirrors vision_patch._save_prediction."""
    p = Path(input_path)
    if p.suffix == ".gz" and p.stem.endswith(".nii"):
        return f"{p.stem[:-4]}_pred.nii.gz"
    if p.suffix == ".nii":
        return f"{p.stem}_pred.nii"
    return f"{p.stem}_pred.nii.gz"


def show_qualitative_examples(plane=2):
    """Show the median-DSC and worst-DSC validation cases for the first model with results on disk."""
    picked = None
    for model_key, entry in MODELS.items():
        results_dir = Path(entry["results_dir"])
        fold_dirs = [d for d in sorted(results_dir.glob("fold_*"))
                     if (d / "results.csv").exists()]
        if fold_dirs:
            picked = (model_key, results_dir, fold_dirs)
            break
    if picked is None:
        print("No fold_*/results.csv found for any model. Run the sweep first.")
        return
    model_key, results_dir, fold_dirs = picked

    # Each row is tagged with its validation fold, which names the predictions/ subdir.
    frames = []
    for fold_dir in fold_dirs:
        df = pd.read_csv(fold_dir / "results.csv")
        df["fold"] = int(fold_dir.name.split("_")[1])
        frames.append(df)
    pooled = pd.concat(frames, ignore_index=True)
    print(f"Model '{model_key}': pooled {len(pooled)} cases from folds "
          f"{sorted(pooled['fold'].unique().tolist())}.")

    # Middle row is the median case, first is the worst: real cases, no interpolation.
    ordered = pooled.sort_values("dsc").reset_index(drop=True)
    picks = [("median DSC", ordered.iloc[len(ordered) // 2]),
             ("worst DSC", ordered.iloc[0])]

    shown = set()
    for label, row in picks:
        case_id = row["case_id"]
        if case_id in shown:                 # median == worst when only one case
            continue
        shown.add(case_id)

        sub = train_df[train_df["case_id"] == case_id]
        if sub.empty:
            print(f"[skip] {case_id}: not found in train_df.")
            continue
        img_path = str(sub.iloc[0]["t1_img_path"])
        gt_path = str(sub.iloc[0]["t1_seg_path"])

        fold = int(row["fold"])
        pred_path = results_dir / f"fold_{fold}" / "predictions" / _pred_filename(img_path)
        if not pred_path.exists():
            print(f"[skip] {case_id}: prediction not found at {pred_path}.")
            continue

        # Default loader (no reorder/resample): input, GT and prediction share the case's original
        # grid. voxel_size is the native spacing, so the display aspect is correct.
        img = MedImage.create(img_path)
        gt = MedMask.create(gt_path)
        pred = MedMask.create(str(pred_path))
        vsize = tio.ScalarImage(img_path).spacing

        dsc = float(row["dsc"])
        print(f"[{label}] {case_id} (fold {fold}): DSC = {dsc:.4f}")
        show_segmentation_comparison(img, gt, pred, metric_value=dsc, metric_name="DSC",
                                     anatomical_plane=plane, voxel_size=vsize)
        plt.show()


show_qualitative_examples()

## 13. Viewing runs and next steps

Every fold logs metrics, parameters, the train/validation split, and model artifacts to MLflow under the per-model experiments `vs5f_unet`, `vs5f_dynunet`, and `vs5f_segmamba`. To browse them, launch the fastMONAI MLflow UI from a notebook cell:

    mlflow_ui = MLflowUIManager()
    mlflow_ui.start_ui()   # opens http://localhost:5001

Artifacts written to disk:

- `preprocessed/` - reoriented, resampled, normalized volumes (shared by all folds).
- `inference_patch_config.json` - patch/preprocessing config, also logged to each fold's MLflow run under `config/`; notebook 02 loads it for inference parity.
- `cv_results_<model>/fold_N/` - per-fold `results.csv`, `benchmark_meta.json`, and NIfTI predictions.
- `cv_results_<model>/cv_summary.csv` - all folds concatenated for one model.
- `cv_model_comparison.csv` - the headline mean +/- std table across models.

`EMACheckpoint` writes each fold's best weights to `models/<best_fname>.pth`. All five folds of a model share that one filename, so the local file ends up holding only the last fold trained. The per-fold weights and exported learners are kept in each fold's MLflow run instead, tagged with its fold number, which is what `find_fold_learners` reads.

To run a trained model on new cases, see `02_inference_new_cases.ipynb`. It loads `inference_patch_config.json` so inference reuses the exact preprocessing contract, and recovers the five fold learners from MLflow to run them as a soft-vote ensemble.

## References

- Connor, S., et al. (2025). The Real-World Impact of Vestibular Schwannoma Fully Automated Volume Measures on the Evaluation of Size Change and Clinical Management Outcomes in a Multidisciplinary Meeting Setting. *Journal of International Advanced Otology*. https://doi.org/10.5152/iao.2025.241693
- Dhayalan, D., et al. (2023). Upfront Radiosurgery vs a Wait-and-Scan Approach for Small- or Medium-Sized Vestibular Schwannoma: The V-REX Randomized Clinical Trial. *JAMA*. https://doi.org/10.1001/jama.2023.12222
- Dorent, R., et al. (2023). CrossMoDA 2021 challenge: Benchmark of cross-modality domain adaptation techniques for vestibular schwannoma and cochlea segmentation. *Medical Image Analysis*. https://doi.org/10.1016/j.media.2022.102628
- Häußler, S. M., et al. (2025). Automatic Segmentation of Vestibular Schwannoma From MRI Using Two Cascaded Deep Learning Networks. *The Laryngoscope*. https://doi.org/10.1002/lary.31979
- Isensee, F., et al. (2024). nnU-Net Revisited: A Call for Rigorous Validation in 3D Medical Image Segmentation. *MICCAI 2024*. https://doi.org/10.1007/978-3-031-72114-4_47
- Kujawa, A., et al. (2024). Deep learning for automatic segmentation of vestibular schwannoma: a retrospective study from multi-center routine MRI. *Frontiers in Computational Neuroscience*. https://doi.org/10.3389/fncom.2024.1365727
- Shapey, J., et al. (2021). Segmentation of vestibular schwannoma from MRI, an open annotated dataset and baseline algorithm. *Scientific Data*. https://doi.org/10.1038/s41597-021-01064-w